In [ ]:
# ✅ Step 1: Setup
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import time
import json
import random
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout

# ✅ Step 2: Download COCO dataset captions & images
!wget http://images.cocodataset.org/zips/train2014.zip
!wget http://images.cocodataset.org/annotations/annotations_trainval2014.zip
!unzip -q train2014.zip -d ./images/
!unzip -q annotations_trainval2014.zip

# ✅ Step 3: Load annotations
annotation_file = './annotations/captions_train2014.json'
with open(annotation_file, 'r') as f:
    annotations = json.load(f)

# Extract image paths and captions
all_captions = []
all_img_names = []

for annot in annotations['annotations']:
    caption = '<start> ' + annot['caption'] + ' <end>'
    image_id = annot['image_id']
    img_name = 'images/train2014/COCO_train2014_' + str(image_id).zfill(12) + '.jpg'
    all_captions.append(caption)
    all_img_names.append(img_name)

# Use a small subset (e.g. 30,000)
train_captions, img_names = zip(*list(zip(all_captions, all_img_names))[:30000])

# ✅ Step 4: Preprocess Images (extract features from CNN)
image_model = InceptionV3(include_top=False, weights='imagenet')
new_input = image_model.input
hidden_layer = image_model.layers[-1].output
image_features_extract_model = tf.keras.Model(new_input, hidden_layer)

# Save preprocessed features to .npy files
def load_image(image_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (299, 299))
    img = preprocess_input(img)
    return img, image_path

# Cache image features
encode_train = sorted(set(img_names))
image_dataset = tf.data.Dataset.from_tensor_slices(encode_train)
image_dataset = image_dataset.map(load_image).batch(16)

os.makedirs('./features', exist_ok=True)
for img, path in tqdm(image_dataset):
    batch_features = image_features_extract_model(img)
    batch_features = tf.reshape(batch_features, (batch_features.shape[0], -1, batch_features.shape[3]))
    for bf, p in zip(batch_features, path):
        path_name = p.numpy().decode("utf-8").split("/")[-1].split('.')[0]
        np.save(f'./features/{path_name}', bf)

# ✅ Step 5: Tokenize captions
tokenizer = Tokenizer(num_words=5000, oov_token="<unk>", filters='!"#$%&()*+.,-/:;=?@[\]^_`{|}~ ')
tokenizer.fit_on_texts(train_captions)
train_seqs = tokenizer.texts_to_sequences(train_captions)
tokenizer.word_index['<pad>'] = 0
tokenizer.index_word[0] = '<pad>'
cap_vector = pad_sequences(train_seqs, padding='post')
max_length = max(len(t) for t in cap_vector)

# ✅ Step 6: Create tf.data.Dataset
img_name_vector, cap_vector = list(img_names), cap_vector.tolist()
img_name_train, img_name_val, cap_train, cap_val = train_test_split(img_name_vector, cap_vector, test_size=0.2)

BATCH_SIZE = 64
BUFFER_SIZE = 1000
embedding_dim = 256
units = 512
vocab_size = len(tokenizer.word_index) + 1
features_shape = 2048
attention_features_shape = 64

def map_func(img_name, cap):
    img_tensor = np.load(f"./features/{img_name.numpy().decode().split('/')[-1].split('.')[0]}.npy")
    return img_tensor, cap

def tf_map_func(img_name, cap):
    return tf.py_function(map_func, [img_name, cap], [tf.float32, tf.int32])

train_dataset = tf.data.Dataset.from_tensor_slices((img_name_train, cap_train))
train_dataset = train_dataset.map(tf_map_func, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

# ✅ Step 7: Define Attention Layer
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V = Dense(1)

    def call(self, features, hidden):
        hidden_with_time_axis = tf.expand_dims(hidden, 1)
        score = tf.nn.tanh(self.W1(features) + self.W2(hidden_with_time_axis))
        attention_weights = tf.nn.softmax(self.V(score), axis=1)
        context_vector = attention_weights * features
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

# ✅ Step 8: CNN Encoder
class CNN_Encoder(tf.keras.Model):
    def __init__(self, embedding_dim):
        super().__init__()
        self.fc = Dense(embedding_dim)

    def call(self, x):
        x = self.fc(x)
        x = tf.nn.relu(x)
        return x

# ✅ Step 9: RNN Decoder with Attention
class RNN_Decoder(tf.keras.Model):
    def __init__(self, embedding_dim, units, vocab_size):
        super().__init__()
        self.units = units
        self.embedding = Embedding(vocab_size, embedding_dim)
        self.lstm = LSTM(units, return_sequences=True, return_state=True)
        self.fc1 = Dense(units)
        self.fc2 = Dense(vocab_size)
        self.attention = BahdanauAttention(units)

    def call(self, x, features, hidden):
        context_vector, attention_weights = self.attention(features, hidden)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        output, state, _ = self.lstm(x)
        x = self.fc1(output)
        x = tf.reshape(x, (-1, x.shape[2]))
        x = self.fc2(x)
        return x, state, attention_weights

    def reset_state(self, batch_size):
        return tf.zeros((batch_size, self.units))

# ✅ Step 10: Training Setup
encoder = CNN_Encoder(embedding_dim)
decoder = RNN_Decoder(embedding_dim, units, vocab_size)

optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    return tf.reduce_mean(loss_ * mask)

@tf.function
def train_step(img_tensor, target):
    loss = 0
    hidden = decoder.reset_state(batch_size=target.shape[0])
    dec_input = tf.expand_dims([tokenizer.word_index['<start>']] * target.shape[0], 1)

    with tf.GradientTape() as tape:
        features = encoder(img_tensor)
        for i in range(1, target.shape[1]):
            predictions, hidden, _ = decoder(dec_input, features, hidden)
            loss += loss_function(target[:, i], predictions)
            dec_input = tf.expand_dims(target[:, i], 1)

    total_loss = loss / int(target.shape[1])
    trainable_variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, trainable_variables)
    optimizer.apply_gradients(zip(gradients, trainable_variables))
    return loss, total_loss

# ✅ Step 11: Training Loop
EPOCHS = 1  # Increase for real training
for epoch in range(EPOCHS):
    total_loss = 0
    for batch, (img_tensor, target) in enumerate(train_dataset):
        batch_loss, t_loss = train_step(img_tensor, target)
        total_loss += t_loss
        if batch % 100 == 0:
            print(f'Epoch {epoch+1} Batch {batch} Loss {batch_loss.numpy():.4f}')
    print(f'Epoch {epoch+1} Loss {total_loss/len(train_dataset):.6f}')


--2025-05-29 08:43:27--  http://images.cocodataset.org/zips/train2014.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 3.5.25.181, 3.5.24.151, 16.182.107.33, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|3.5.25.181|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13510573713 (13G) [application/zip]
Saving to: ‘train2014.zip’

train2014.zip       100%[===================>]  12.58G  56.0MB/s    in 4m 13s  

2025-05-29 08:47:40 (51.0 MB/s) - ‘train2014.zip’ saved [13510573713/13510573713]

--2025-05-29 08:47:40--  http://images.cocodataset.org/annotations/annotations_trainval2014.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 3.5.10.223, 54.231.192.241, 3.5.20.157, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|3.5.10.223|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 252872794 (241M) [application/zip]
Saving to: ‘annotations_trainval2014.zip’

annotations_trainva 100

100%|██████████| 382/382 [33:22<00:00,  5.24s/it]


Epoch 1 Batch 0 Loss 104.0714
Epoch 1 Batch 100 Loss 57.2447
Epoch 1 Batch 200 Loss 47.8243
Epoch 1 Batch 300 Loss 43.0420
Epoch 1 Loss 1.045120


In [ ]:
# ✅ Step 12: Inference (Caption Generation)
def evaluate(image):
    attention_plot = np.zeros((max_length, attention_features_shape))
    hidden = decoder.reset_state(batch_size=1)

    temp_input = tf.expand_dims(load_image(image)[0], 0)
    img_tensor_val = image_features_extract_model(temp_input)
    img_tensor_val = tf.reshape(img_tensor_val, (img_tensor_val.shape[0], -1, img_tensor_val.shape[3]))
    features = encoder(img_tensor_val)

    dec_input = tf.expand_dims([tokenizer.word_index['<start>']], 0)
    result = []

    for i in range(max_length):
        predictions, hidden, attention_weights = decoder(dec_input, features, hidden)
        attention_plot[i] = tf.reshape(attention_weights, (-1,)).numpy()
        predicted_id = tf.argmax(predictions[0]).numpy()
        predicted_word = tokenizer.index_word.get(predicted_id, '<unk>')
        if predicted_word == '<end>':
            return result, attention_plot
        result.append(predicted_word)
        dec_input = tf.expand_dims([predicted_id], 0)
    return result, attention_plot

# ✅ Step 13: Visualize Caption and Attention
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def plot_attention(image, result, attention_plot):
    temp_image = np.array(Image.open(image))
    fig = plt.figure(figsize=(10, 10))
    len_result = len(result)
    for l in range(len_result):
        temp_att = np.resize(attention_plot[l], (8, 8))
        ax = fig.add_subplot(len_result//2, len_result//2, l+1)
        ax.set_title(result[l])
        img = ax.imshow(temp_image)
        ax.imshow(temp_att, cmap='gray', alpha=0.6, extent=img.get_extent())
    plt.tight_layout()
    plt.show()

# ✅ Step 14: Test on an Image
image_path = 'images/train2014/COCO_train2014_000000581929.jpg'  # make sure this exists
description, attention_plot = evaluate(image_path)
print('Prediction Caption:', ' '.join(description))
plot_attention(image_path, description, attention_plot)
